# whisper-serving — run it yourself

Transcribe real audio with Whisper, then reproduce two of the measurements from the repo:
the WER gate that guards CI, and the batching speedup.

Runs on Colab's free T4. Takes about 5 minutes.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
# No output means CPU-only. Runtime -> Change runtime type -> T4 GPU, then rerun.

In [ ]:
!git clone -q https://github.com/riya0920/asr-serving-vllm-k8s.git
%cd asr-serving-vllm-k8s
!pip install -q 'datasets<4' soundfile librosa httpx 2>&1 | tail -1
print('ready')

## 1. Build a golden set

Clips of exactly 30 seconds from LibriSpeech, with reference transcripts. Whole utterances
only, silence-padded to the encoder window.

An earlier version of this script cut audio at 30s mid-utterance while dropping that
utterance from the reference. Whisper transcribed the fragment correctly and every word
scored as an insertion, which reported 15% WER for a model that was actually at 1.6%.

In [ ]:
!python bench/build_golden.py --clips 5 --out golden 2>&1 | tail -8

In [ ]:
import json, IPython.display as d
m = json.load(open('golden/manifest.json'))
c = m['clips'][0]
print(f"{c['file']}  {c['seconds']}s  ({c['speech_seconds']}s speech, {c['utterances']} utterances)\n")
print('reference:', c['reference'][:300])
d.display(d.Audio(f"golden/audio/{c['file']}"))

## 2. Transcribe it

In [ ]:
import time, torch, soundfile as sf
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL = 'openai/whisper-large-v3-turbo'
proc = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to('cuda').eval()

wav, sr = sf.read(f"golden/audio/{c['file']}", dtype='float32')
t0 = time.perf_counter()
with torch.inference_mode():
    ids = model.generate(
        proc(wav, sampling_rate=sr, return_tensors='pt').input_features.to('cuda', torch.float16),
        max_new_tokens=440)
torch.cuda.synchronize()
hyp = proc.batch_decode(ids, skip_special_tokens=True)[0]
print(f'{(time.perf_counter()-t0)*1000:.0f} ms for 30s of audio\n')
print(hyp)

## 3. Score it — the gate that guards CI

Corpus WER is total errors over total reference words, not the mean of per-clip rates.
Averaging per-clip lets one short clip with two errors swing the number as hard as a long
clip with fifty, which makes the gate flaky for reasons unrelated to the model.

The error breakdown matters as much as the total. Substitutions are the model being wrong.
A pile of deletions and insertions usually means your references are misaligned.

In [ ]:
import sys; sys.path.insert(0, 'bench')
from wer import wer, normalize

r = wer(c['reference'], hyp)
print(f"WER  {r['wer']:.4f}   ({r['substitutions']}S {r['deletions']}D {r['insertions']}I "
      f"over {r['ref_words']} words)\n")
print('normalized reference: ', normalize(c['reference'])[:160])
print('normalized hypothesis:', normalize(hyp)[:160])

## 4. Why batching matters

Whisper decodes autoregressively, and each step moves the whole model through memory to
produce one token. That leaves compute idle. Batching fills it with other requests.

Below: the same clips one at a time, then together.

In [ ]:
import numpy as np

waves = [sf.read(f"golden/audio/{x['file']}", dtype='float32')[0] for x in m['clips']]
feats = proc(waves, sampling_rate=16000, return_tensors='pt').input_features.to('cuda', torch.float16)

with torch.inference_mode():  # warm up so the first call is not timed
    model.generate(feats[:1], max_new_tokens=440)
torch.cuda.synchronize()

t0 = time.perf_counter()
with torch.inference_mode():
    for i in range(len(waves)):
        model.generate(feats[i:i+1], max_new_tokens=440)
torch.cuda.synchronize()
seq = time.perf_counter() - t0

t0 = time.perf_counter()
with torch.inference_mode():
    model.generate(feats, max_new_tokens=440)
torch.cuda.synchronize()
bat = time.perf_counter() - t0

n = len(waves)
print(f'one at a time   {seq:6.2f}s   {n/seq:5.2f} clips/s')
print(f'batched         {bat:6.2f}s   {n/bat:5.2f} clips/s')
print(f'                          {seq/bat:5.2f}x')

That is static batching, and it is the weaker version. Every clip in the batch waits for the
longest transcript in it, so a 33-token clip holds its slot while a 117-token clip finishes.

vLLM does continuous batching instead: it admits and evicts at every decode step, so a
finished sequence frees its slot immediately. That is worth 7.4x over sequential in the repo's
measurements, against 21.8x once batch slots and a distilled decoder are added.

## What the repo measures that this notebook cannot

A single Colab GPU cannot show the parts that need a cluster or a load generator running for
minutes:

| measurement | result |
|---|---|
| throughput, 4 stages | 0.60 → 13.07 req/s (21.8x) |
| p99 at concurrency 1 | 473 ms |
| KEDA reaction to an 8x spike | under 2 s, 2 → 16 replicas |
| canary auto-rollback | 22 s |
| a faster GPU going slower | H100 5.93 vs A40 13.07 req/s |

Raw artifacts for all of them are in `results/`, and the reasoning is in `docs/`.

The autoscaling and delivery halves run without a GPU at all — `bash scripts/kind_up.sh`
brings up k3s, Prometheus and KEDA against a stub engine that publishes the same metrics.